# Practica: ETL conceptual y datos para Power BI (Tema 4)

**Asignatura:** GAD-2401

## Objetivos
1. Simular un mini-ETL con pandas (equivalente mental a Power Query).
2. Construir tablas de hechos y dimensiones (modelo estrella).
3. Exportar CSV listos para importar en **Power BI Desktop**.
4. Calcular KPIs en pandas como referencia para validar medidas DAX.

> Completa los ejercicios del HTML en el cuaderno y luego abre Power BI Desktop.


## 1. Librerias

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
OUT = Path("datos_powerbi")
OUT.mkdir(exist_ok=True)
print("OK – carpeta:", OUT.resolve())

## 2. Dimensiones: producto, cliente, fecha

In [ ]:
productos = pd.DataFrame({
    "ProductoID": [1, 2, 3, 4, 5],
    "Producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Headset"],
    "Categoria": ["Computo", "Accesorio", "Accesorio", "Computo", "Accesorio"],
    "CostoUnitario": [12000, 250, 600, 3500, 800]
})

clientes = pd.DataFrame({
    "ClienteID": range(1, 21),
    "Cliente": [f"Cliente_{i:02d}" for i in range(1, 21)],
    "Region": np.random.choice(["Norte", "Sur", "Centro", "Occidente"], 20),
    "Segmento": np.random.choice(["Retail", "Mayoreo"], 20, p=[0.7, 0.3])
})

fechas = pd.DataFrame({"Fecha": pd.date_range("2024-01-01", "2025-12-31", freq="D")})
fechas["Anio"] = fechas["Fecha"].dt.year
fechas["Mes"] = fechas["Fecha"].dt.month
fechas["NombreMes"] = fechas["Fecha"].dt.month_name()
fechas["Trimestre"] = fechas["Fecha"].dt.quarter
fechas["DiaSemana"] = fechas["Fecha"].dt.day_name()

display(productos)
display(clientes.head())
display(fechas.head())

## 3. Tabla de hechos (ventas)

In [ ]:
n = 500
hechos = pd.DataFrame({
    "VentaID": range(1, n + 1),
    "Fecha": np.random.choice(fechas["Fecha"], n),
    "ProductoID": np.random.choice(productos["ProductoID"], n),
    "ClienteID": np.random.choice(clientes["ClienteID"], n),
    "Cantidad": np.random.randint(1, 6, n)
})
hechos = hechos.merge(productos[["ProductoID", "CostoUnitario"]], on="ProductoID")
# Precio con margen
hechos["PrecioUnitario"] = (hechos["CostoUnitario"] * np.random.uniform(1.2, 1.6, n)).round(2)
hechos["Importe"] = (hechos["Cantidad"] * hechos["PrecioUnitario"]).round(2)
hechos = hechos.drop(columns=["CostoUnitario"])
display(hechos.head())
print(hechos.shape)

## 4. Mini-ETL (equivalente Power Query)

Simulamos datos "sucios" y los limpiamos: tipos, nulos, columnas.


In [ ]:
sucio = hechos.copy()
sucio.loc[0:4, "Importe"] = np.nan
sucio["RegionExtra"] = "borrar"  # columna innecesaria

# Transformaciones
limpio = sucio.copy()
limpio["Importe"] = limpio["Importe"].fillna(limpio["Cantidad"] * limpio["PrecioUnitario"])
limpio = limpio.drop(columns=["RegionExtra"])
limpio["Fecha"] = pd.to_datetime(limpio["Fecha"])
print("Nulos en Importe despues de limpiar:", limpio["Importe"].isna().sum())
print("Columnas finales:", list(limpio.columns))

## 5. KPIs de referencia (para validar DAX en Power BI)

In [ ]:
ventas_totales = limpio["Importe"].sum()
tickets = limpio["VentaID"].nunique()
ticket_promedio = ventas_totales / tickets
por_categoria = (
    limpio.merge(productos, on="ProductoID")
    .groupby("Categoria", as_index=False)["Importe"].sum()
    .sort_values("Importe", ascending=False)
)
print(f"Ventas totales: ${ventas_totales:,.2f}")
print(f"Tickets: {tickets}")
print(f"Ticket promedio: ${ticket_promedio:,.2f}")
display(por_categoria)

## 6. Exportar CSV para Power BI Desktop

In [ ]:
limpio.to_csv(OUT / "hechos_ventas.csv", index=False)
productos.to_csv(OUT / "dim_producto.csv", index=False)
clientes.to_csv(OUT / "dim_cliente.csv", index=False)
fechas.to_csv(OUT / "dim_fecha.csv", index=False)
print("Archivos exportados en:", OUT.resolve())
for f in sorted(OUT.glob("*.csv")):
    print(" -", f.name)

## 7. Guia rapida en Power BI Desktop

1. **Obtener datos** → Texto/CSV → cargar los 4 archivos.
2. En Power Query: verificar tipos (`Fecha` como fecha, IDs como entero, `Importe` como decimal).
3. **Cerrar y aplicar**.
4. Vista de modelo: relacionar
   - `hechos_ventas[ProductoID]` → `dim_producto[ProductoID]`
   - `hechos_ventas[ClienteID]` → `dim_cliente[ClienteID]`
   - `hechos_ventas[Fecha]` → `dim_fecha[Fecha]`
5. Marcar `dim_fecha` como tabla de fechas.
6. Crear medidas:
   - `Ventas = SUM(hechos_ventas[Importe])`
   - `Tickets = DISTINCTCOUNT(hechos_ventas[VentaID])`
   - `Ticket Promedio = DIVIDE([Ventas], [Tickets])`
7. Construir 3 paginas de informe (resumen, por region/categoria, tendencia temporal).
8. Compara los totales con los KPIs de la seccion 5 de este notebook.


## 8. Ejercicios del notebook

### A
Agrega una columna `Margen` aproximado en hechos (usa CostoUnitario de productos) y recalcula un KPI de margen total.

### B
Filtra solo region Norte en pandas y calcula ventas; luego reproduce el mismo filtro en Power BI.

### C
Escribe 5 lineas del documento del proyecto final (pregunta de negocio + hallazgo esperado).


### Espacio A

In [ ]:
# Tu codigo


### Espacio B y C

B)

C)


## 9. Cierre del tema 4 (y del curso tematico 2–4)

| Tema | Enfoque |
|------|---------|
| 2 | Correlacion y asociacion |
| 3 | Visualizacion y proyecciones |
| 4 | Power BI: ETL, modelo, informes, proyecto |

**Fin de la practica del tema 4**
